In [7]:
import pandas as pd
import numpy as np
import re, math, unicodedata
from pathlib import Path

IN_PATH  = Path("KITCODE Negatif.xlsx")
OUT_PATH = Path("TCD_KITCODE_HT.xlsx")

def norm_cols(cols):
    return [re.sub(r"\s+", " ", str(c).strip()) for c in cols]

def strip_accents_lower(s):
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.lower()

def find_col_by_regex(df, patterns):
    for pat in patterns:
        for c in df.columns:
            if re.search(pat, c, flags=re.IGNORECASE):
                return c
    return None

def find_operation_col_containing(df, token_a="annulation", token_b="avoir"):
    for c in df.columns:
        series = df[c].astype(str).map(strip_accents_lower)
        if (series.str.contains(token_a) & series.str.contains(token_b)).any():
            return c
    return None

def to_float(x):
    try:
        if isinstance(x, str):
            x = x.replace(" ", "").replace("\u202f", "").replace(",", ".")
        return float(x)
    except Exception:
        return np.nan

df = pd.read_excel(IN_PATH, engine="openpyxl")
df.columns = norm_cols(df.columns)

kit_col = find_col_by_regex(df, [r"\bKIT[_\s]?CODE\b", r"^KITCODE$", r"^CODE[_\s]?KIT$"])
amount_col = find_col_by_regex(
    df,
    [r"^MONTANT[_\s]?HT$", r"\bMONTANT\b.*\bHT\b", r"TOTAL.*HT", r"AMOUNT.*(HT|EX\s*VAT|NET)"]
)
operation_col = find_operation_col_containing(df, "annulation", "avoir")

missing = []
if not kit_col:      missing.append("KITCODE")
if not amount_col:   missing.append("MONTANT_HT")
if not operation_col:missing.append("colonne texte contenant 'ANNULATION' et 'AVOIR'")
if missing:
    raise ValueError(f"Colonnes introuvables : {missing}\nColonnes dispo : {list(df.columns)}")

df["_op_norm"] = df[operation_col].map(strip_accents_lower)
df["_is_cancel_avoir"] = df["_op_norm"].str.contains("annulation") & df["_op_norm"].str.contains("avoir")
df["_amount"] = df[amount_col].apply(to_float)

sum_annul = df.loc[df["_is_cancel_avoir"]].groupby(kit_col)["_amount"].sum().rename("MONTANT_HT (ANNULATION AVEC AVOIR)")
sum_other = df.loc[~df["_is_cancel_avoir"]].groupby(kit_col)["_amount"].sum().rename("MONTANT_HT (autres opérations)")
cnt_annul = df.loc[df["_is_cancel_avoir"]].groupby(kit_col).size().rename("Nombre d'annulations")
cnt_other = df.loc[~df["_is_cancel_avoir"]].groupby(kit_col).size().rename("Nombre d'autres opérations")

tcd = (
    pd.concat([sum_annul, sum_other, cnt_annul, cnt_other], axis=1)
      .fillna({
          "MONTANT_HT (ANNULATION AVEC AVOIR)": 0.0,
          "MONTANT_HT (autres opérations)": 0.0,
          "Nombre d'annulations": 0,
          "Nombre d'autres opérations": 0
      })
      .reset_index()
      .rename(columns={kit_col: "KITCODE"})
      .sort_values("KITCODE")
)

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as writer:
    tcd.to_excel(writer, index=False, sheet_name="TCD")